# Week 21: High-Performance Inference & Compilers
## Notebook 08: CUDA Shared Memory Tiling & Matrix Multiplication

### 1. Conceptual Intuition & Theoretical Background
In standard naive GPU matrix multiplication ($C = A \times B$), every thread loads an entire row of $A$ and column of $B$ directly from high-latency global memory (DRAM). For $N \times N$ matrices:
$$\text{Global Memory Accesses} = 2 N^3 \text{ reads}$$
Global memory bandwidth saturates rapidly, leaving Tensor Cores idle.

**Shared Memory Tiling (Block Tiling):**
CUDA GPUs have fast on-chip **Shared Memory (SRAM)** ($approx 100\times$ faster than DRAM).
1. Matrices are partitioned into $B \times B$ tiles (e.g. $16 \times 16$ or $32 \times 32$).
2. All threads in a thread block cooperatively load a tile into shared memory:
$$\text{Cooperatively Loaded Elements} = 2 B^2$$
3. Threads synchronize via barrier (`__syncthreads()`).
4. Threads compute tile products from shared memory, reusing each value $B$ times.
$$\text{DRAM Accesses} = \frac{2 N^3}{B}$$

**Arithmetic Intensity:**
$$\text{Arithmetic Intensity} = \frac{2 B^3 \text{ FLOPs}}{2 B^2 \times 4 \text{ Bytes}} = \frac{B}{4} \text{ FLOPs / Byte}$$
Increasing tile size $B$ scales arithmetic intensity linearly until limited by SRAM capacity.

---

### 2. Scratch Python Implementation: Tiled Shared Memory Simulation

In [ ]:
import numpy as np

def naive_matmul(A, B):
    """Simulates naive thread evaluation accessing global memory on every step."""
    N = A.shape[0]
    C = np.zeros((N, N))
    global_reads = 0
    for i in range(N):
        for j in range(N):
            for k in range(N):
                C[i, j] += A[i, k] * B[k, j]
                global_reads += 2  # 2 reads from slow DRAM
    return C, global_reads

def tiled_shared_memory_matmul(A, B, tile_size=4):
    """Simulates CUDA block-level shared memory tiling."""
    N = A.shape[0]
    C = np.zeros((N, N))
    num_tiles = N // tile_size
    global_reads = 0
    
    # Iterate over matrix tiles
    for bi in range(num_tiles):
        for bj in range(num_tiles):
            # Tile accumulator in local registers
            tile_C = np.zeros((tile_size, tile_size))
            
            for bk in range(num_tiles):
                # 1. Cooperative load into on-chip shared memory
                s_A = A[bi*tile_size : (bi+1)*tile_size, bk*tile_size : (bk+1)*tile_size].copy()
                s_B = B[bk*tile_size : (bk+1)*tile_size, bj*tile_size : (bj+1)*tile_size].copy()
                global_reads += 2 * (tile_size * tile_size)
                
                # 2. Compute inner product entirely inside on-chip SRAM
                tile_C += np.dot(s_A, s_B)
                
            C[bi*tile_size : (bi+1)*tile_size, bj*tile_size : (bj+1)*tile_size] = tile_C
            
    return C, global_reads

# Benchmark Matrix Dimensions (N = 16)
N = 16
tile_size = 4
A_mat = np.random.randn(N, N)
B_mat = np.random.randn(N, N)

res_naive, reads_naive = naive_matmul(A_mat, B_mat)
res_tiled, reads_tiled = tiled_shared_memory_matmul(A_mat, B_mat, tile_size=tile_size)

print("--- CUDA Shared Memory Tiling Simulation ---")
print(f"Matrix Dimension: {N} x {N} | Tile Size: {tile_size} x {tile_size}")
print(f"Naive DRAM Global Reads:  {reads_naive}")
print(f"Tiled DRAM Global Reads:  {reads_tiled}")
print(f"Global DRAM Traffic Reduction: {(1 - reads_tiled / reads_naive) * 100:.1f}%")
print(f"Max Absolute Verification Difference: {np.max(np.abs(res_naive - res_tiled)):.1e}")

### 3. Step-by-Step Code Walkthrough
1. **Cooperative Shared Memory Load**: Threads load a compact sub-matrix into on-chip SRAM.
2. **Inner Dot Product**: Computations execute at SRAM speed without round-trips to high-latency DRAM.
3. **Arithmetic Intensity Scaling**: Decreases slow DRAM memory traffic by a factor of $\text{tile\_size}$.

---

### 4. Industry Applications & Trade-offs
- **FlashAttention & TensorRT Matrix Kernels:** Tiling enables FlashAttention to compute attention without writing $N \times N$ attention matrices to DRAM.
- **Trade-off:** Shared memory is limited (e.g. 100KB per SM); large tile sizes restrict thread occupancy.

---

### 5. Top Interview Questions & Answers
**Q1: What is a Shared Memory Bank Conflict in CUDA?**
> **Answer:** CUDA shared memory is partitioned into 32 independent banks. If multiple threads in a warp access different memory addresses within the same bank simultaneously, accesses are serialized, reducing throughput.

**Q2: How does memory tiling increase Arithmetic Intensity?**
> **Answer:** By loading a tile into shared memory once and reusing it $B$ times for computations, the ratio of math operations to DRAM byte reads increases by a factor of $B$.